<a href="https://colab.research.google.com/github/JaiJanarious/steam-gaming-for-investors/blob/main/Gaming_For_Investors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Gaming for investors App.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Mounted at /content/drive
✅ Loaded 1000 games from: /content/drive/MyDrive/MIST 4600/


In [3]:
def top_devs_by_owners(games, n=5):
    return (games.groupby("developers")["owners_low"]
                 .sum()
                 .sort_values(ascending=False)
                 .head(n))

print(top_devs_by_owners(games))

developers
Valve                     338000000
Arrowhead Game Studios     54000000
Pocketpair                 51000000
Respawn                    50500000
Amazon Games               50000000
Name: owners_low, dtype: int64


In [20]:
import requests

url = "https://steamspy.com/api.php?request=top100forever"
data = requests.get(url).json()

df = pd.DataFrame(data.values())
df = df.sort_values("average_forever", ascending=False).reset_index(drop=True)
df.index += 1  # start ranking at 1

# owners comes back as text like "50,000,000 .. 100,000,000", so turn it into one number
parts = df["owners"].str.replace(",", "").str.split(" .. ", expand=True)
df["owners_num"] = (parts[0].astype(int) + parts[1].astype(int)) / 2

# scale each column to 0-1 (biggest value = 1), then average the three
cols = ["ccu", "average_forever", "owners_num"]
df["average_score"] = (df[cols] / df[cols].max()).mean(axis=1)

# rank by the new score, highest first
df = df.sort_values("average_score", ascending=False).reset_index(drop=True)
df.index += 1

counts = {}
totals = {}

for appid, info in developers.items():
    dev = info["developer"]
    if dev in counts:
        counts[dev] += 1
        totals[dev] += info["score"]
    else:
        counts[dev] = 1
        totals[dev] = info["score"]

# keep developers with 2+ games and calculate their average score
averages = {}
for dev in counts:
    if counts[dev] > 1:
        averages[dev] = {
            "num_games": counts[dev],
            "avg_score": round(totals[dev] / counts[dev], 3),
        }

avg_table = pd.DataFrame.from_dict(averages, orient="index")
avg_table.index.name = "developer"
avg_table = avg_table.sort_values("avg_score", ascending=False)
avg_table


,num_games,avg_score
developer,,
Valve,12,0.198
Facepunch Studios,2,0.166
Bohemia Interactive,2,0.083
CD PROJEKT RED,2,0.073
"Firaxis Games, Aspyr (Mac), Aspyr (Linux)",2,0.057
"FromSoftware, Inc.",3,0.056
"CAPCOM Co., Ltd.",2,0.050
Bethesda Game Studios,2,0.048
Endnight Games Ltd,2,0.034


In [21]:
for game in df.to_dict("records"):
    developers[game["appid"]] = {
        "developer": game["developer"],
        "score": round(game["average_score"], 3),
    }
dev_table = pd.DataFrame.from_dict(developers, orient="index")
dev_table.index.name = "appid"
dev_table

,developer,score
appid,,
570,Valve,0.892
730,Valve,0.857
440,Valve,0.297
1172470,Respawn,0.275
252490,Facepunch Studios,0.231
...,...,...
333930,Splash Damage,0.011
203160,"Crystal Dynamics, Eidos-Montréal, Feral Intera...",0.011
224260,No More Room in Hell Team,0.011
